In [ ]:
# ============================================================
# install
# ============================================================
import sys, subprocess, os, time, base64, json, re, threading
from datetime import datetime, timezone

try:
    from google.colab import userdata  # Colab-only
    api_key = userdata.get("UI_OPENAI_KEY")
except Exception:
    api_key = None

if not api_key:
    print("Нет OPENAI_API_KEY в Colab Secrets")
    raise SystemExit(0)

def pip_install(pkgs):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q"] + pkgs)

pip_install([
    "fastapi>=0.110.0",
    "uvicorn[standard]>=0.27.0",
    "python-multipart>=0.0.9",
    "openai>=1.40.0",
    "requests>=2.31.0",
])

# cloudflared (Quick Tunnel) does NOT require ngrok authtoken
subprocess.check_call([
    "bash", "-lc",
    "set -euo pipefail; "
    "if ! command -v cloudflared >/dev/null 2>&1; then "
    "  wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O /usr/local/bin/cloudflared; "
    "  chmod +x /usr/local/bin/cloudflared; "
    "fi"
])

# ============================================================
# server
# ============================================================
from fastapi import FastAPI, UploadFile, File, Form, HTTPException
from fastapi.responses import JSONResponse, Response
from fastapi.middleware.cors import CORSMiddleware
from pydantic import BaseModel
from openai import OpenAI
import requests
from urllib.parse import urlencode

client = OpenAI(api_key=api_key)

APP_PORT = 8000

app = FastAPI(title="AI-Cook Backend (Lovable)", version="1.0.0")

app.add_middleware(
    CORSMiddleware,
    allow_origins=["*"],
    allow_credentials=True,
    allow_methods=["*"],
    allow_headers=["*"],
)

# In-memory sessions (lives only while this Colab runtime is running)
SESSIONS = {}

def now_iso():
    return datetime.now(timezone.utc).isoformat()

def get_session(session_id: str):
    if session_id not in SESSIONS:
        SESSIONS[session_id] = {
            "profile": {"likes": "", "allergies": ""},
            "full_history": [],  # [{"role":"user"/"assistant","content":str,"ts":iso}]
            "fridge_context": {
                "lists": [],  # [{"label":"fridge_1","items":[...],"last_updated_at":iso}]
                "active_index": None,
            },
            "recommendations_state": {
                "last_options": [],         # [{"id":"1","title":"...","why":"..."}]
                "selected_option": "",
                "rejected_options": [],     # ["1", "2", ...] or titles
            },
            "assistant_messages_store": {
                "next_id": 1,
                "messages": {},  # {"1": {"text": "...", "ts": iso}}
            },
            "client_history_raw": "",
        }
    return SESSIONS[session_id]

def store_assistant_message(sess, text: str):
    mid = str(sess["assistant_messages_store"]["next_id"])
    sess["assistant_messages_store"]["next_id"] += 1
    sess["assistant_messages_store"]["messages"][mid] = {"text": text, "ts": now_iso()}
    return mid

def append_history(sess, role: str, content: str):
    sess["full_history"].append({"role": role, "content": content, "ts": now_iso()})

def b64_data_url(img_bytes: bytes, mime: str):
    return f"data:{mime};base64,{base64.b64encode(img_bytes).decode('utf-8')}"

def parse_state_block(text: str):
    """
    Looks for <state>{json}</state>. Returns (clean_text, state_dict_or_None).
    """
    m = re.search(r"<state>\s*(\{.*?\})\s*</state>", text, flags=re.DOTALL)
    if not m:
        return text.strip(), None
    raw = m.group(1).strip()
    clean = (text[:m.start()] + text[m.end():]).strip()
    try:
        state = json.loads(raw)
    except Exception:
        state = None
    return clean, state

def safe_str(x):
    if x is None:
        return ""
    if isinstance(x, (list, dict)):
        return json.dumps(x, ensure_ascii=False)
    return str(x)

def vision_extract_items(image_bytes: bytes):
    """
    Returns (list[str] or None, raw_text)
    """
    data_url = b64_data_url(image_bytes, "image/jpeg")
    prompt = (
        "Ты помощник, который извлекает продукты из фото холодильника.\n"
        "Верни ТОЛЬКО одну строку в одном из форматов:\n"
        "1) ITEMS: item1, item2, item3, ...  (максимум 30 позиций)\n"
        "2) UNSURE: краткая причина (если фото размыто/темно/не видно)\n"
        "Правила: не выдумывай. Пиши продукты коротко, по-русски."
    )
    try:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": "Ты очень точный vision-ассистент по продуктам. Не выдумывай."},
                {"role": "user", "content": [
                    {"type": "text", "text": prompt},
                    {"type": "image_url", "image_url": {"url": data_url}},
                ]},
            ],
            temperature=0.2,
        )
        out = (resp.choices[0].message.content or "").strip()
        if out.upper().startswith("UNSURE"):
            return None, out
        if out.upper().startswith("ITEMS:"):
            items_str = out.split(":", 1)[1].strip()
            items = [i.strip(" -•\t\r\n") for i in items_str.split(",")]
            items = [i for i in items if i]
            if len(items) < 2:
                return None, "UNSURE: мало распознаваемых продуктов"
            return items[:30], out
        return None, "UNSURE: неверный формат ответа распознавания"
    except Exception as e:
        return None, f"UNSURE: ошибка распознавания ({type(e).__name__})"

def strip_image_links(text: str):
    t = text or ""
    t = re.sub(r"!\[[^\]]*\]\([^)]+\)", "", t)
    t = re.sub(r"https?://\S+", "", t)
    t = re.sub(r"\n{3,}", "\n\n", t).strip()
    return t

def guess_dish_title_from_state_or_text(sess, state, assistant_text):
    if isinstance(state, dict):
        dt = safe_str(state.get("dish_title")).strip()
        if dt:
            return dt

    selected = ""
    if isinstance(state, dict):
        selected = safe_str(state.get("selected_option")).strip()
    if not selected:
        selected = safe_str(sess["recommendations_state"].get("selected_option")).strip()

    last_options = sess["recommendations_state"].get("last_options") or []
    if selected and isinstance(last_options, list):
        for o in last_options:
            if isinstance(o, dict) and safe_str(o.get("id")).strip() == selected:
                title = safe_str(o.get("title")).strip()
                if title:
                    return title

    at = (assistant_text or "").strip()
    if at:
        first_line = at.splitlines()[0].strip()
        first_line = re.sub(r"^(Рецепт|Блюдо|Выбрали|Выбор)\s*[:\-]\s*", "", first_line, flags=re.IGNORECASE).strip()
        if 3 <= len(first_line) <= 90:
            return first_line

    return "выбранное блюдо"

def needs_recipe_image(state, assistant_text):
    if isinstance(state, dict) and safe_str(state.get("dish_image_prompt")).strip():
        return True
    t = (assistant_text or "").lower()
    has_ingredients = ("ингредиент" in t) or ("что нужно" in t)
    has_steps = ("шаг" in t) or ("приготов" in t) or ("инструкц" in t)
    return has_ingredients and has_steps

def _clean_query_title(title: str) -> str:
    t = (title or "").strip()
    t = re.sub(r"\s+", " ", t)
    t = re.sub(r"[“”\"'`]", "", t)
    t = re.sub(r"\(.*?\)", "", t).strip()
    return t[:120] if t else "dish"

def _wikimedia_commons_search_file_title(query: str, timeout=10):
    """
    Returns a Commons file title like 'File:Something.jpg' or ''.
    """
    base = "https://commons.wikimedia.org/w/api.php"
    params = {
        "action": "query",
        "list": "search",
        "srsearch": query,
        "srnamespace": 6,  # File namespace
        "srlimit": 5,
        "format": "json",
    }
    headers = {"User-Agent": "ColabAIBackend/1.0 (Lovable)"}
    r = requests.get(base, params=params, headers=headers, timeout=timeout)
    r.raise_for_status()
    data = r.json()
    hits = (((data or {}).get("query") or {}).get("search") or [])
    if not hits:
        return ""
    title = (hits[0].get("title") or "").strip()
    return title if title.lower().startswith("file:") else ""

def _wikimedia_commons_get_image_url(file_title: str, timeout=10):
    """
    Returns a direct image URL (prefers thumburl) or ''.
    """
    if not file_title:
        return ""
    base = "https://commons.wikimedia.org/w/api.php"
    params = {
        "action": "query",
        "titles": file_title,
        "prop": "imageinfo",
        "iiprop": "url|mime",
        "iiurlwidth": 1024,
        "format": "json",
    }
    headers = {"User-Agent": "ColabAIBackend/1.0 (Lovable)"}
    r = requests.get(base, params=params, headers=headers, timeout=timeout)
    r.raise_for_status()
    data = r.json()
    pages = ((data or {}).get("query") or {}).get("pages") or {}
    for _, page in pages.items():
        ii = page.get("imageinfo")
        if ii and isinstance(ii, list) and ii:
            info = ii[0]
            return (info.get("thumburl") or info.get("url") or "").strip()
    return ""

def fetch_dish_image_via_web(dish_title: str, session_id: str = ""):
    """
    Интернет-поиск картинки (Wikimedia Commons), скачивание и конвертация в data URL.
    Возвращает data URL или "".
    Логи:
      - trigger: dish title + source
      - success: size
      - error: exception type + empty
    """
    source = "wikimedia_commons"
    title = _clean_query_title(dish_title)
    print(f"[img] session_id={session_id!r} trigger=final_recipe title={title!r} source={source}")

    try:
        # 1) Search file
        file_title = _wikimedia_commons_search_file_title(title)
        if not file_title:
            # fallback: add "dish"
            file_title = _wikimedia_commons_search_file_title(f"{title} dish")
        if not file_title:
            print(f"[img] session_id={session_id!r} result=FAIL reason=no_search_results dish_image=''")
            return ""

        # 2) Resolve to direct image URL
        img_url = _wikimedia_commons_get_image_url(file_title)
        if not img_url:
            print(f"[img] session_id={session_id!r} result=FAIL reason=no_image_url dish_image=''")
            return ""

        # 3) Download
        headers = {"User-Agent": "ColabAIBackend/1.0 (Lovable)"}
        t0 = time.time()
        r = requests.get(img_url, headers=headers, timeout=20)
        r.raise_for_status()
        img_bytes = r.content or b""
        dt = int((time.time() - t0) * 1000)

        if not img_bytes:
            print(f"[img] session_id={session_id!r} result=FAIL reason=empty_bytes dish_image=''")
            return ""

        ctype = (r.headers.get("Content-Type") or "").split(";")[0].strip().lower()
        if not ctype.startswith("image/"):
            # fallback by URL extension
            if img_url.lower().endswith(".png"):
                ctype = "image/png"
            elif img_url.lower().endswith(".webp"):
                ctype = "image/webp"
            else:
                ctype = "image/jpeg"

        data_url = b64_data_url(img_bytes, ctype)
        print(f"[img] session_id={session_id!r} result=SUCCESS bytes={len(img_bytes)} content_type={ctype} time_ms={dt}")
        return data_url

    except Exception as e:
        print(f"[img] session_id={session_id!r} result=ERROR error={type(e).__name__} dish_image=''")
        return ""

def llm_chat_response(sess, user_message: str, session_id: str):
    """
    Returns (assistant_text, state_dict_or_None, dish_image_or_None)
      - dish_image: None => not final recipe (do not include field)
                   ""   => final recipe but not found
                   "data:image/...;base64,..." => final recipe found
    """
    profile = sess["profile"]
    fridge_lists = sess["fridge_context"]["lists"]
    rec_state = sess["recommendations_state"]

    recent = sess["full_history"][-16:]
    recent_txt = "\n".join([f'{m["role"].upper()}: {m["content"]}' for m in recent]).strip()

    fridge_repr = []
    for idx, lst in enumerate(fridge_lists):
        label = lst.get("label") or f"fridge_{idx+1}"
        items = lst.get("items") or []
        ts = lst.get("last_updated_at") or ""
        fridge_repr.append(f"- {label} (updated: {ts}): {', '.join(items) if items else '—'}")
    fridge_block = "\n".join(fridge_repr).strip() if fridge_repr else "—"

    sys_prompt = (
        "Ты AI-кулинар «что приготовить из холодильника». Общайся по-русски.\n"
        "Цели: собрать профиль, учитывать ограничения, предлагать блюда из того, что есть.\n"
        "Критично: аллергии/нельзя — жёсткий фильтр. Не предлагай блюда с запрещённым.\n\n"
        "Строго запрещено:\n"
        "- Любые ссылки/URL на изображения или любые картинки в тексте ответа.\n"
        "- Любые markdown-картинки.\n"
        "Изображение блюда возвращается ТОЛЬКО через поле dish_image в API (это делает бэкенд).\n\n"
        "Правила диалога:\n"
        "1) Если профиль ПУСТОЙ (likes пусто И allergies пусто) и в новом сообщении НЕТ явных ответов,\n"
        "   задай РОВНО 2 вопроса одним сообщением:\n"
        "   (а) Что вы любите/предпочтения?\n"
        "   (б) Есть ли аллергии или что нельзя?\n"
        "   НИКАКИХ рецептов и вариантов блюд в этом сообщении.\n"
        "2) Если в сообщении есть предпочтения/аллергии — обнови профиль и НЕ повторяй эти 2 вопроса.\n"
        "3) После заполнения профиля:\n"
        "   - если есть продукты (хотя бы один список fridge_*) — предложи 3 варианта: 1 основной + 2 альтернативы.\n"
        "     Формат коротко: «Вариант 1: ... (почему подходит)».\n"
        "   - если списков нет — попроси фото холодильника.\n"
        "4) Когда пользователь выбирает вариант (может быть «1», «вариант 2», «беру пасту» и т.п.),\n"
        "   выдай полный рецепт выбранного блюда: ингредиенты (что есть/что заменить/что докупить), шаги, время/сложность.\n"
        "   Основывайся на том, что есть, чтобы можно было приготовить сразу без похода в магазин.\n"
        "   В ЭТОМ случае ты ОБЯЗАН добавить <state>{...}</state> и внутри JSON ОБЯЗАН вернуть dish_image_prompt.\n"
        "5) По просьбе «ещё» предлагай новые 3 варианта, не повторяя уже предложенные/отвергнутые.\n"
        "6) Если чего-то не хватает — можно задать 1 уточняющий вопрос (но НЕ на самом первом шаге с 2 вопросами профиля).\n\n"
        "Формат ответа:\n"
        "- Основной ответ — обычный человеческий текст.\n"
        "- В конце МОЖЕШЬ (или ОБЯЗАН, если выдал полный рецепт выбранного блюда) добавить:\n"
        "  <state>{JSON}</state>\n"
        "  где JSON может содержать:\n"
        "  profile_update: {likes: str, allergies: str}\n"
        "  last_options: [{id:'1', title:'...', why:'...'}, ...]  (ровно 3 если предлагал варианты)\n"
        "  selected_option: str\n"
        "  rejected_options: [str, ...]\n"
        "  dish_title: str\n"
        "  dish_image_prompt: str  (ОБЯЗАТЕЛЬНО при полном рецепте выбранного блюда)\n"
        "Важно: не пиши никакого другого «JSON» вне <state>...</state>."
    )

    user_payload = {
        "profile_current": profile,
        "fridge_lists": fridge_lists,
        "fridge_lists_human": fridge_block,
        "recommendations_state": rec_state,
        "recent_history": recent_txt,
        "new_user_message": user_message,
    }

    try:
        resp = client.chat.completions.create(
            model="gpt-4o-mini",
            messages=[
                {"role": "system", "content": sys_prompt},
                {"role": "user", "content": "Контекст (JSON):\n" + json.dumps(user_payload, ensure_ascii=False)},
            ],
            temperature=0.6,
        )
        raw = (resp.choices[0].message.content or "").strip()
        assistant_text, state = parse_state_block(raw)
        assistant_text = strip_image_links(assistant_text)

        if isinstance(state, dict):
            pu = state.get("profile_update")
            if isinstance(pu, dict):
                likes = safe_str(pu.get("likes")).strip()
                allergies = safe_str(pu.get("allergies")).strip()
                if likes:
                    sess["profile"]["likes"] = likes
                if allergies:
                    sess["profile"]["allergies"] = allergies

            lo = state.get("last_options")
            if isinstance(lo, list) and lo:
                norm = []
                for o in lo[:3]:
                    if isinstance(o, dict):
                        oid = safe_str(o.get("id")).strip() or str(len(norm) + 1)
                        title = safe_str(o.get("title")).strip()
                        why = safe_str(o.get("why")).strip()
                        if title:
                            norm.append({"id": oid, "title": title, "why": why})
                if len(norm) == 3:
                    sess["recommendations_state"]["last_options"] = norm

            so = safe_str(state.get("selected_option")).strip()
            if so:
                sess["recommendations_state"]["selected_option"] = so

            ro = state.get("rejected_options")
            if isinstance(ro, list):
                sess["recommendations_state"]["rejected_options"] = [safe_str(x).strip() for x in ro if safe_str(x).strip()]

        # Only if final recipe -> search image on web and return as data URL
        if needs_recipe_image(state, assistant_text):
            dish_title = guess_dish_title_from_state_or_text(sess, state, assistant_text)
            dish_image = fetch_dish_image_via_web(dish_title, session_id=session_id)  # "" if not found
            return assistant_text.strip(), state, dish_image

        print(f"[img] session_id={session_id!r} trigger=no_recipe -> skip")
        return assistant_text.strip(), state, None

    except Exception as e:
        print(f"[chat] session_id={session_id!r} llm_failed: {type(e).__name__}")
        return "Временно не получилось, попробуйте ещё раз.", None, None

# ============================================================
# endpoints
# ============================================================
@app.post("/api/chat")
async def api_chat(
    session_id: str = Form(...),
    user_message: str = Form(""),
    history: str = Form(None),
    image: UploadFile = File(None),
):
    print(f"[chat] incoming: session_id={session_id!r}, has_image={bool(image)}, user_message_len={len(user_message or '')}")

    if not session_id or not session_id.strip():
        raise HTTPException(status_code=400, detail="Пустой session_id")

    user_message = (user_message or "").strip()
    if (not user_message) and (image is None):
        raise HTTPException(status_code=400, detail="Пришлите сообщение или фото")

    sess = get_session(session_id.strip())

    if history is not None:
        sess["client_history_raw"] = history

    # Step A: only if new image arrived
    if image is not None:
        try:
            img_bytes = await image.read()
        except Exception:
            img_bytes = b""

        if not img_bytes:
            return JSONResponse(
                status_code=400,
                content={"session_id": session_id, "assistant_message": "Фото не прочиталось. Пришлите другое, пожалуйста."},
            )

        print("[chat] image received -> extracting items via OpenAI (vision)")
        items, raw_vision = vision_extract_items(img_bytes)
        if not items:
            print(f"[chat] vision unsure: {raw_vision}")
            assistant_text = "Фото получилось не очень понятным. Пришлите, пожалуйста, другое фото холодильника (поближе и при хорошем свете)."
            assistant_text = strip_image_links(assistant_text)
            append_history(sess, "assistant", assistant_text)
            mid = store_assistant_message(sess, assistant_text)
            return JSONResponse(
                content={
                    "session_id": session_id,
                    "assistant_message": assistant_text,
                    "message_id": mid,
                }
            )

        label = f"fridge_{len(sess['fridge_context']['lists']) + 1}"
        sess["fridge_context"]["lists"].append(
            {"label": label, "items": items, "last_updated_at": now_iso()}
        )
        sess["fridge_context"]["active_index"] = len(sess["fridge_context"]["lists"]) - 1
        print(f"[chat] fridge_context updated: {label} items={len(items)}")

    if user_message:
        append_history(sess, "user", user_message)

    print("[chat] calling OpenAI for assistant response")
    assistant_text, state, dish_image = llm_chat_response(sess, user_message, session_id=session_id)

    append_history(sess, "assistant", assistant_text)
    mid = store_assistant_message(sess, assistant_text)

    options_list = None
    lo = sess["recommendations_state"].get("last_options") or []
    if isinstance(lo, list) and len(lo) == 3:
        options_list = lo

    resp = {
        "session_id": session_id,
        "assistant_message": assistant_text,
        "message_id": mid,
    }
    if options_list:
        resp["options_list"] = options_list
    # For final recipe: include dish_image (may be empty string). Otherwise: omit.
    if dish_image is not None:
        resp["dish_image"] = dish_image

    return JSONResponse(content=resp)

class TTSRequest(BaseModel):
    session_id: str
    message_id: str | None = None
    text: str | None = None
    voice: str | None = "alloy"
    format: str | None = "mp3"

def content_type_for_audio(fmt: str):
    f = (fmt or "mp3").lower().strip()
    if f == "mp3":
        return "audio/mpeg"
    if f == "wav":
        return "audio/wav"
    if f == "aac":
        return "audio/aac"
    if f in ("opus", "ogg"):
        return "audio/ogg"
    return "application/octet-stream"

@app.post("/api/tts")
async def api_tts(req: TTSRequest):
    print(f"[tts] incoming: session_id={req.session_id!r}, message_id={req.message_id!r}, has_text={bool(req.text)}")

    if not req.session_id or not req.session_id.strip():
        raise HTTPException(status_code=400, detail="Пустой session_id")

    text = (req.text or "").strip()
    if not text:
        print(f"[tts] session_id={req.session_id!r} result=400 reason=no_text")
        raise HTTPException(status_code=400, detail="нечего озвучивать")

    voice = (req.voice or "alloy").strip() or "alloy"
    response_format = (req.format or "mp3").lower().strip() or "mp3"

    try:
        print(f"[tts] session_id={req.session_id!r} start voice={voice} format={response_format} text_len={len(text)}")
        t0 = time.time()
        with client.audio.speech.with_streaming_response.create(
            model="gpt-4o-mini-tts",
            voice=voice,
            input=text,
            response_format=response_format,
        ) as r:
            audio_bytes = r.read()
        dt = int((time.time() - t0) * 1000)
        print(f"[tts] session_id={req.session_id!r} done ok=True bytes={len(audio_bytes)} time_ms={dt}")
        return Response(content=audio_bytes, media_type=content_type_for_audio(response_format))
    except Exception as e:
        print(f"[tts] session_id={req.session_id!r} done ok=False error={type(e).__name__}")
        raise HTTPException(status_code=500, detail="Временно не получилось, попробуйте ещё раз")

# ============================================================
# tunnel (cloudflared quick tunnel, no authtoken needed)
# ============================================================
def start_uvicorn():
    import uvicorn
    uvicorn.run(app, host="0.0.0.0", port=APP_PORT, log_level="info")

server_thread = threading.Thread(target=start_uvicorn, daemon=True)
server_thread.start()
time.sleep(1.5)

cloudflared_proc = subprocess.Popen(
    [
        "cloudflared", "tunnel",
        "--url", f"http://127.0.0.1:{APP_PORT}",
        "--no-autoupdate",
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
    text=True,
    bufsize=1,
)

public_url = None
url_re = re.compile(r"https://[a-z0-9\-]+\.trycloudflare\.com", re.IGNORECASE)

start_time = time.time()
while True:
    line = cloudflared_proc.stdout.readline() if cloudflared_proc.stdout else ""
    if line:
        m = url_re.search(line)
        if m:
            public_url = m.group(0).rstrip("/")
            break
    if cloudflared_proc.poll() is not None:
        break
    if time.time() - start_time > 60:
        break

if not public_url:
    print("Не удалось поднять публичный туннель cloudflared. Проверьте вывод выше.")
    try:
        cloudflared_proc.terminate()
    except Exception:
        pass
    raise SystemExit(1)

# ============================================================
# print
# ============================================================
print("\n" + "=" * 60)
print(f"Public Base URL: {public_url}")
print(f"Chat Endpoint: {public_url}/api/chat")
print(f"TTS Endpoint: {public_url}/api/tts")
print("=" * 60 + "\n")

# ============================================================
# run (keep alive while this cell runs)
# ============================================================
while True:
    time.sleep(3600)


INFO:     Started server process [228]
INFO:     Waiting for application startup.
INFO:     Application startup complete.
INFO:     Uvicorn running on http://0.0.0.0:8000 (Press CTRL+C to quit)



Public Base URL: https://persian-projector-scott-harold.trycloudflare.com
Chat Endpoint: https://persian-projector-scott-harold.trycloudflare.com/api/chat
TTS Endpoint: https://persian-projector-scott-harold.trycloudflare.com/api/tts

[chat] incoming: session_id='0194b1e7-d664-4871-8bf5-a8e994a17ff9', has_image=True, user_message_len=36
[chat] image received -> extracting items via OpenAI (vision)
[chat] fridge_context updated: fridge_1 items=14
[chat] calling OpenAI for assistant response
[img] session_id='0194b1e7-d664-4871-8bf5-a8e994a17ff9' trigger=no_recipe -> skip
INFO:     209.50.51.178:0 - "POST /api/chat HTTP/1.1" 200 OK
[chat] incoming: session_id='0194b1e7-d664-4871-8bf5-a8e994a17ff9', has_image=False, user_message_len=33
[chat] calling OpenAI for assistant response
[img] session_id='0194b1e7-d664-4871-8bf5-a8e994a17ff9' trigger=no_recipe -> skip
INFO:     209.50.51.178:0 - "POST /api/chat HTTP/1.1" 200 OK
[chat] incoming: session_id='0194b1e7-d664-4871-8bf5-a8e994a17ff9', 

KeyboardInterrupt: 